# Per-process resource reference

A lookup table for what each nf-core/mag process costs: how much CPU it keeps busy, how
much memory it peaks at, how long it runs, and how much work-directory space it leaves
behind. Median and observed range per process.

These are **consumed** resources, not the CPUs and memory the pipeline reserves. Ranges
pool both datasets and every assembly a process ran on, so they describe the spread to
expect in general rather than the effect of any single input.

Columns, all per task: `peak_rss_gb` is the memory high-water mark, the figure a memory
limit has to clear; `realtime_h` is wall-clock; `workdir_gb` is what the task leaves in
the work directory; `cpu_hours` is CPU time actually consumed, so a task pinning 8 cores
for an hour counts as 8.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import polars as pl

from mag_bench.build import prepare
from mag_bench.resources import process_resources

pl.Config.set_tbl_rows(100)
pl.Config.set_tbl_width_chars(200)

frames = prepare("../data")
resources = process_resources(frames["trace"])

## Summary table

One row per process, grouped by pipeline stage and ordered within each stage by total CPU
time. `granularity` says what a single task represents, which is what turns a per-task
median into a per-sample cost: a *per bin* process runs hundreds of times per sample.


In [2]:
resources.select(pl.exclude("cpu_hours_total"))

stage,process,granularity,n_tasks,cpu_hours,cpu_hours_range,peak_rss_gb,peak_rss_gb_range,realtime_h,realtime_h_range,workdir_gb,workdir_gb_range
enum,str,str,u32,f64,str,f64,str,f64,str,f64,str
"""Short-read preprocessing""","""BOWTIE2_PHIX_REMOVAL_ALIGN""","""per sample""",11,10.03,"""6.42-32.08""",0.09,"""0.06-0.13""",1.35,"""0.87-4.45""",5.31,"""3.11-14.26"""
"""Short-read preprocessing""","""FASTP""","""per sample""",11,0.26,"""0.15-0.66""",1.4,"""1.4-1.4""",0.07,"""0.04-0.14""",5.72,"""3.36-15.43"""
"""Short-read preprocessing""","""FASTQC_RAW""","""per sample""",11,0.14,"""0.08-0.36""",2.4,"""1.7-2.9""",0.08,"""0.04-0.2""",0.0,"""0.0-0.0"""
"""Short-read preprocessing""","""FASTQC_TRIMMED""","""per sample""",11,0.13,"""0.08-0.38""",2.3,"""1.9-2.6""",0.07,"""0.04-0.2""",0.0,"""0.0-0.0"""
"""Short-read preprocessing""","""BOWTIE2_PHIX_REMOVAL_BUILD""","""once per run""",2,0.0,"""0.0-0.0""",0.0,"""0.0-0.0""",0.0,"""0.0-0.0""",0.01,"""0.01-0.01"""
"""Long-read preprocessing""","""PORECHOP_ABI""","""per sample""",11,0.85,"""0.37-3.38""",33.2,"""13.8-57.9""",0.85,"""0.36-2.09""",8.18,"""3.32-15.19"""
"""Long-read preprocessing""","""CHOPPER""","""per sample""",11,0.76,"""0.28-1.53""",0.02,"""0.01-0.07""",0.66,"""0.24-1.42""",7.49,"""2.76-14.85"""
"""Long-read preprocessing""","""NANOPLOT_RAW""","""per sample""",11,0.13,"""0.05-0.18""",0.81,"""0.59-1.3""",0.14,"""0.06-0.2""",0.0,"""0.0-0.0"""
"""Long-read preprocessing""","""NANOPLOT_FILTERED""","""per sample""",11,0.11,"""0.04-0.17""",0.53,"""0.42-0.65""",0.12,"""0.05-0.18""",0.0,"""0.0-0.0"""


## Heaviest processes

Total CPU time over both runs, as a quick answer to what dominates a full-featured run.


In [3]:
resources.select("process", "stage", "granularity", "n_tasks", "cpu_hours", "cpu_hours_total").sort(
    "cpu_hours_total", descending=True
).head(15)

process,stage,granularity,n_tasks,cpu_hours,cpu_hours_total
str,enum,str,u32,f64,f64
"""PROKKA""","""Annotation (Prodigal/Prokka)""","""per bin""",50086,0.08,4141.5
"""COMEBIN_RUNCOMEBIN""","""Binning""","""per assembly""",55,31.78,3063.5
"""BUSCO_BUSCO""","""Bin QC (BUSCO)""","""per assembly x binner""",385,4.63,1958.7
"""GUNC_RUN""","""Bin QC (GUNC)""","""per assembly x binner""",385,3.08,1217.4
"""CATPACK_BINS""","""Taxonomic classification (CAT)""","""per assembly x binner""",35,23.95,881.7
"""METASPADESHYBRID""","""Hybrid assembly""","""per sample""",11,54.76,672.7
"""CONCOCT_CONCOCT""","""Binning""","""per assembly""",55,8.38,525.0
"""METASPADES""","""Short-read assembly""","""per sample""",11,36.01,518.4
"""CHECKM2_PREDICT""","""Bin QC (CheckM2)""","""per assembly x binner""",385,1.03,416.6


## Export


In [4]:
path = Path("../data/process_resources.csv")
resources.write_csv(path)
print(f"{path}: {resources.height} processes")

../data/process_resources.csv: 65 processes
